# 12.09 · Ship it: UI, CI/CD, safe releases, and a public deployment

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 4–5 h |
| 📋 Prerequisites | Lessons 12.04–12.07 (the service, its image, its SLOs), 10.08 (the eval gate). |
| 🔑 Keys | none for the 🧪 cells (optional: a GitHub account, a Render account, a Neon account and `GROQ_API_KEY` for the 🌐 public deployment) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Serve** a small browser UI from the service that streams runs, survives a page reload, and can stop a run.
2. **Protect** a public demo: no demo keys, a global daily cost cap, a kill switch, and a `/version` endpoint.
3. **Build** a CI/CD pipeline: lint → tests → an image tagged with the commit → smoke test → the eval gate → deploy.
4. **Release safely**: shadow comparison, a sticky canary, and rollback criteria written *before* the release.
5. **Practise a rollback** and time it.
6. **Deploy** the service to a free tier (Render + Neon) and know what "free" does not give you.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [MDN: Content Security Policy](https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/CSP) | Why the UI loads only its own scripts |
| [GitHub: publishing Docker images](https://docs.github.com/en/actions/tutorials/publish-packages/publish-docker-images) · [Container registry](https://docs.github.com/en/packages/working-with-a-github-packages-registry/working-with-the-container-registry) | Pushing to GHCR with `GITHUB_TOKEN` |
| [GitHub: protected branches](https://docs.github.com/en/repositories/configuring-branches-and-merges-in-your-repository/managing-protected-branches/about-protected-branches) · [environments](https://docs.github.com/en/actions/how-tos/deploy/configure-and-manage-deployments/manage-environments) | Required checks; reviewers before a deploy |
| [pre-commit](https://pre-commit.com/) · [nbstripout](https://github.com/kynan/nbstripout) | Hooks before every commit |
| [SRE workbook: canarying releases](https://sre.google/workbook/canarying-releases/) | Canary size, duration, and comparing against a control |
| [Render: free instances](https://render.com/docs/free) · [deploy an image](https://render.com/docs/deploying-an-image) · [Blueprint spec](https://render.com/docs/blueprint-spec) | Spin-down, ephemeral disk, `render.yaml` |
| [Render: deploy hooks](https://render.com/docs/deploy-hooks) · [rollbacks](https://render.com/docs/rollbacks) | Deploying a given image tag from CI; rolling back |
| [Neon pricing](https://neon.com/pricing) | The free Postgres plan's limits |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4" "fastapi>=0.141.1,<0.142" "uvicorn>=0.54.0,<1" "pydantic-settings>=2.16.0,<3" "httpx>=0.28.1,<1" "pyjwt[crypto]>=2.15.1,<3" "langgraph-checkpoint-postgres>=3.2.0,<4" "psycopg[binary]>=3.3.6,<4" "prometheus-client>=0.26.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("fastapi", "uvicorn", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-ship-it'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · A UI that survives a reload

`apps/m12_service/ui/` is three static files (`index.html`, `app.js`, `style.css`) that the service serves at `/ui/`: no framework, no build step, no second server. Why not Streamlit or Gradio? Both are good for internal demos, but they run their own Python server, so a public demo would need a second service, a second login, and a second thing to keep alive on a free tier. Here the UI is a client of the same API you already secured.

What `app.js` does:

- **Stream** with `fetch` and read the SSE body itself (`EventSource` can't send an `Authorization` header).
- **Survive a reload**: it keeps the run id and the last event id in `sessionStorage`; after a reload it reconnects with `Last-Event-ID` and gets only the events it missed (Lesson 12.04's resumable stream).
- **Stop** a run (`POST /runs/{id}/cancel`), and send an `Idempotency-Key` with each message so a double click doesn't start two runs.
- Show answers with `textContent`, never `innerHTML`: model output is untrusted text.

The service sends a strict **Content-Security-Policy** with the UI: only scripts from this origin, no inline scripts, no framing.

In [ ]:
# 🧪 Offline: the service with its UI, in this notebook (stopped at the end of the lesson).
import httpx

from mastery.service.app import create_app, serve_in_background
from mastery.service.desk import support_manager
from mastery.service.settings import ServiceSettings

UI_DIR = REPO_ROOT / "apps" / "m12_service" / "ui"
CUSTOMER, MANAGER = {"Authorization": "Bearer demo-customer"}, {"Authorization": "Bearer demo-manager"}
app = create_app(lambda s: support_manager(s, delay_s=0.3), ServiceSettings(release="1.4.0", log_level="ERROR"), ui_dir=UI_DIR)
BASE, stop_service = serve_in_background(app)
client = httpx.Client(base_url=BASE, timeout=30)
page = client.get("/ui/")
print("open in your browser:", f"{BASE}/ui/  (key: demo-customer)")
print("UI     :", page.status_code, "·", len(page.text), "bytes")
print("CSP    :", page.headers["content-security-policy"][:60], "…")
print("version:", client.get("/version").json())

**You should now see** the URL to open (try it: sign in with `demo-customer`, ask *"How do I rotate an API key?"*, reload the page while it answers), `UI : 200`, the start of the CSP, and `version: {'release': '1.4.0', 'mode': 'demo', 'model': …, 'paused': False}`.

The next cell does what the browser does on a reload: read part of a stream, drop the connection, reconnect with `Last-Event-ID`.

In [ ]:
# 🧪 Offline: a "reload" in the middle of a run.
import json

def read_events(run_id, last=None, limit=None):
    headers = CUSTOMER | ({"Last-Event-ID": str(last)} if last is not None else {})
    events = []
    with client.stream("GET", f"/runs/{run_id}/stream", headers=headers) as response:
        block = {}
        for raw in response.iter_lines():
            if raw.startswith(":"):
                continue
            if not raw:
                if block:
                    events.append(block)
                    block = {}
                    if limit and len(events) >= limit:
                        break                          # the tab was reloaded
                continue
            field, _, value = raw.partition(":")
            block[field] = value.strip()
    return events

thread = client.post("/threads", headers=CUSTOMER).json()["thread_id"]
run = client.post(f"/threads/{thread}/runs", json={"message": "How do I rotate an API key?"}, headers=CUSTOMER).json()
before = read_events(run["run_id"], limit=2)
after = read_events(run["run_id"], last=before[-1]["id"])
print("before reload:", [(e["id"], e["event"]) for e in before])
print("after reload :", [(e["id"], e["event"]) for e in after][:4], "…", after[-1]["event"])
print("no duplicates:", not {e["id"] for e in before} & {e["id"] for e in after})

### What just happened?

- The first connection got events `0` and `1` and was dropped, like a tab being reloaded. The run kept going on the server.
- The second connection sent `Last-Event-ID: 1` and got everything from `2` to `end`, with no duplicates. `app.js` does exactly this with the id it saved in `sessionStorage`.

---
## Part 2 · Protecting a public demo

A public URL is found by bots within hours. Four guards, all in `ServiceSettings`:

| Guard | Setting | What it stops |
|---|---|---|
| Lock-down check | `MASTERY_SERVICE_PUBLIC=true` | Starting with the demo keys (anyone can read them in this repo), keys under 24 characters, a JWT secret under 32 |
| Per-user and global daily cost caps | `BUDGET_PER_USER_DAY_USD`, `BUDGET_GLOBAL_DAY_USD` | One user, or everyone together, spending your Groq quota (429) |
| Kill switch | `MASTERY_SERVICE_PAUSED`, or `POST /admin/pause` (managers) | Abuse or a bad answer in production: new runs get 503 at once, no redeploy |
| `/version` | `MASTERY_SERVICE_RELEASE` | "Which version is live?" answered in one request (CI uses it to confirm a deploy) |

In [ ]:
# 🧪 Offline: the public lock-down refuses unsafe settings at startup.
import secrets

for label, kwargs in [("demo keys", {}), ("short key", {"api_keys": "abc=C-1002:customer"})]:
    try:
        ServiceSettings(public=True, **kwargs)
        print(f"{label:<10}: started?!")
    except ValueError as exc:
        print(f"{label:<10}: refused ·", str(exc).split("public=True but: ")[-1][:70])
strong = f"{secrets.token_urlsafe(24)}=C-1002:customer,{secrets.token_urlsafe(24)}=ops:manager"
print("strong keys:", ServiceSettings(public=True, api_keys=strong).public)

**You should now see** two refusals (*still contains a demo key*, *at least 24 characters*) and `strong keys: True`. The keys were generated with `secrets.token_urlsafe` and never printed: in the real deployment they live in the platform's secret store.

In [ ]:
# 🧪 Offline: the kill switch, then the global daily cost cap.
t = client.post("/threads", headers=CUSTOMER).json()["thread_id"]
print("customer pause :", client.post("/admin/pause", json={"paused": True}, headers=CUSTOMER).status_code)
print("manager pause  :", client.post("/admin/pause", json={"paused": True}, headers=MANAGER).json())
r = client.post(f"/threads/{t}/runs", json={"message": "hi"}, headers=CUSTOMER)
print("run while paused:", r.status_code, r.json()["detail"], "· Retry-After", r.headers["retry-after"])
client.post("/admin/pause", json={"paused": False}, headers=MANAGER)

capped = create_app(support_manager, ServiceSettings(budget_global_day_usd=0.0015, log_level="ERROR"))
capped_url, stop_capped = serve_in_background(capped)
with httpx.Client(base_url=capped_url, timeout=30) as c:
    codes = []
    for i in range(6):
        tid = c.post("/threads", headers=CUSTOMER).json()["thread_id"]
        resp = c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"}, headers=CUSTOMER)
        codes.append(resp.status_code)
        if resp.status_code == 202:
            c.get(f"/runs/{resp.json()['run_id']}/stream", headers=CUSTOMER)   # wait for it to finish
    print("global cap $0.0015:", codes, "·", resp.json().get("detail", "")[:60])
stop_capped()

### What just happened?

- Only a manager can flip the kill switch; while it is on, new runs get **503** with `Retry-After: 60`, and `/version` shows `paused: True`. `POST /admin/pause` changes this one process; set `MASTERY_SERVICE_PAUSED=true` in the platform to keep it after a restart.
- With a global cap of $0.0015 a day: `[202, 202, 202, 429, 429, 429]` and *global daily cap reached*. Each run is admitted only if today's spend plus its **upper-bound estimate** (about $0.0012 here) fits under the cap; after it finishes it is charged its real cost (about $0.00013 each in DEMO). After three runs the estimate no longer fits. On the public demo, the cap is what stands between a bot and your free-tier quota: set it to what you are willing to lose in a day.

---
## Part 3 · CI/CD: from a pull request to a running release

Three workflows in `.github/workflows/`:

| Workflow | When | What |
|---|---|---|
| `ci.yml` (Module 00) | every push and PR | unit tests on Linux and Windows, every notebook headless |
| `eval-gate.yml` (Lesson 10.08) | PRs that touch the agent | the release suite and its gates; exit 1 blocks the merge |
| `m12-service.yml` (this lesson) | PRs and pushes that touch the service | lint → service tests → image tagged with the commit → smoke test → (main) push to GHCR → deploy through Render's deploy hook → wait until `/version` shows the new commit |

Make `Eval gate` and `m12 service` **required checks** on `main` (Settings → Branches), and add reviewers to the `production` environment if you want a human to approve each deploy. `.pre-commit-config.yaml` runs nbstripout (keeping cell ids), the same ruff gate, a YAML check and a private-key detector before every commit.

In [ ]:
# 🧪 Offline: read the pipeline, then run the eval gate the way CI does.
import subprocess
import sys

import yaml

flow = yaml.safe_load((REPO_ROOT / ".github" / "workflows" / "m12-service.yml").read_text(encoding="utf-8"))
for job, spec in flow["jobs"].items():
    print(f"{job:<7} needs={spec.get('needs', '-'):<5}", [s.get("name") or s.get("uses") for s in spec["steps"]][:6])

env = {**os.environ, "PYTHONPATH": str(SRC_DIR)}
for candidate in ("baseline", "no-auth-context"):
    done = subprocess.run([sys.executable, "-m", "mastery.evals.release_gate", "--candidate", candidate],
                          capture_output=True, text=True, encoding="utf-8", cwd=REPO_ROOT, env=env)
    verdict = [line for line in done.stdout.splitlines() if "blocked by" in line or "all gates passed" in line][-1:]
    print(f"gate {candidate:<16}: exit {done.returncode}", verdict)

**You should now see** the three jobs (`test`, `image` needing `test`, `deploy` needing `image`) with their steps, then `gate baseline: exit 0` (*all gates passed*) and `gate no-auth-context: exit 1` with *blocked by: red team*. In CI that exit code is what turns the check red.

Why tag the image with the commit (`:${{ github.sha }}`) and not `:latest`? A tag that never moves is a release you can name, compare and **roll back to**. `latest` is whatever was pushed last.

---
## Part 4 · Safe releases: shadow, canary, rollback criteria

A release is any change in code, prompt, model id or config: each one can change answers. `mastery.service.release` has three tools.

**Shadow:** replay the same inputs through stable and candidate, compare the outcomes; users only see stable's answers. We replay the release cases and the red-team attacks through the shipped desk and through `no-auth-context`, the refactor that "forgot" the signed-in customer (Lesson 10.08).

In [ ]:
# 🧪 Offline: shadow comparison of two desk versions.
from mastery.agents.release_suite import BASELINE, NO_AUTH_CONTEXT, REDTEAM, release_cases
from mastery.service.release import CanaryRouter, RollbackCriteria, canary_decision, shadow_compare

cases = release_cases(40)
rows = []
for candidate in (BASELINE, NO_AUTH_CONTEXT):
    result = shadow_compare(BASELINE, candidate, cases, attacks=REDTEAM)
    rows.append({"candidate": candidate.name, "cases": result["cases"], "both_pass": result["both_pass"],
                 "regressions": result["regressions"], "attacks": result["attacks"],
                 "newly_unblocked": ",".join(result["newly_unblocked"]) or "-", "ship": result["ship"]})
table(rows, ["candidate", "cases", "both_pass", "regressions", "attacks", "newly_unblocked", "ship"])

**You should now see** `baseline` against itself with 0 regressions and `ship True`, and `no-auth-context` with the same task results but four attacks newly let through (`RT01,RT02,RT03,RT07`) and `ship False`. Shadow traffic found a security regression that the ordinary cases didn't show.

**Canary:** a small, *sticky* share of users gets the candidate. Sticky matters for agents: a thread that switches versions mid-conversation mixes two prompts in one checkpoint. The decision compares the canary's SLIs and cost with stable's, using the 95% interval so one unlucky error doesn't roll back a good release.

In [ ]:
# 🧪 Offline (SIMULATED traffic): route users, then decide on three canaries.
from mastery.service.slo import synthetic_runs

router = CanaryRouter(0.05, salt="release-1.5.0")
users = [f"C-{i:04d}" for i in range(2000)]
print("canary share:", sum(router.route(u) == "canary" for u in users) / len(users),
      "· sticky:", all(router.route(u) == router.route(u) for u in users))

criteria = RollbackCriteria(min_runs=50, max_sli_drop=0.01, max_cost_ratio=1.5)   # written BEFORE the release
stable = synthetic_runs(500, seed=1)
canaries = {"healthy (60 runs)": synthetic_runs(60, seed=2),
            "too early (20 runs)": synthetic_runs(20, seed=2),
            "slow provider (60 runs)": synthetic_runs(600, seed=3, incident=True)[-60:]}
for name, runs in canaries.items():
    decision = canary_decision(stable, runs, criteria)
    print(f"{name:<24} → {decision.action:<8}", decision.reasons[0][:80])

### What just happened?

- 5% of users (about 100 of 2,000) go to the canary, always the same ones; a new `salt` picks a new slice for the next release.
- A healthy canary is **promoted**; 20 runs are too few to judge (**hold**); the canary during a provider slowdown is **rolled back** because its TTFT and availability are confidently worse than stable's.
- The criteria were written before the release. During an incident nobody should be arguing about thresholds.

> On one free-tier instance there is no load balancer to split traffic. The same router works inside the service (pick the graph by `router.route(user_id)`), or you run the candidate as a second service on its own URL and send shadow traffic to it.

---
## Part 5 · Practise a rollback

Release `1.5.0` ships with a bug (DEMO: 30% of model calls fail). We watch it, apply the criteria, and roll back to `1.4.0`, timing each step. On Render, "roll back" means deploying the previous image tag (the deploy hook with `imgURL=…:<previous sha>`) or clicking *Rollback* on the previous deploy.

In [ ]:
# 🧪 Offline: deploy a bad release, detect it, roll back. Timings are this notebook's, not Render's.
import time

from mastery.service.slo import RunRecord

def deploy(release, fail_rate):
    managers = {}

    def build(settings):
        managers["m"] = support_manager(settings, fail_rate=fail_rate, seed=7)
        return managers["m"]

    url, stop = serve_in_background(create_app(build, ServiceSettings(release=release, max_concurrent=8, log_level="ERROR")))
    return url, stop, managers

def traffic(url, n=60):
    with httpx.Client(base_url=url, timeout=30) as c:
        for i in range(n):
            tid = c.post("/threads", headers=CUSTOMER).json()["thread_id"]
            run_id = c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"}, headers=CUSTOMER).json()["run_id"]
            c.get(f"/runs/{run_id}/stream", headers=CUSTOMER)

clock = time.perf_counter()
url, stop, m = deploy("1.4.0", 0.0)
traffic(url)
baseline_runs = m["m"].records()
stop()
url, stop, m = deploy("1.5.0", 0.3)
traffic(url)
detected = time.perf_counter()
decision = canary_decision(baseline_runs, m["m"].records(), RollbackCriteria(min_runs=50))
print("1.5.0 decision:", decision.action, "·", decision.reasons[0][:70])
stop()
url, stop, m = deploy("1.4.0", 0.0)                 # the previous image tag
with httpx.Client(base_url=url, timeout=10) as c:
    print("live again    :", c.get("/version").json()["release"], "· ready", c.get("/ready").status_code)
print(f"rollback took {time.perf_counter() - detected:.1f} s after the decision")
stop()

**You should now see** `1.5.0 decision: rollback · availability: canary 0.250 …`: 30% of *model calls* fail, and each run makes several, so about three runs in four fail. Then `live again : 1.4.0 · ready 200`, well under a second after the decision.

### What just happened?

The decision took the data you already collect (`RunRecord`s, Lesson 12.01) and the criteria you wrote in advance. The rollback is fast because it is a **redeploy of an image that already exists**, not a rebuild. On Render expect one to two minutes (pull the image, start, pass `/ready`); the goal of the drill is *under five minutes from decision to healthy*. Write the time in your runbook (`apps/m12_service/RUNBOOK.md`), and remember a prompt or model change rolls back the same way only if it is part of the image or the config, not edited live.

---
## Part 6 · 🌐 Deploy it: Render + Neon, free tiers (checked 2026-10-16)

| Piece | Free tier | Watch out |
|---|---|---|
| **Render** web service (from the GHCR image) | 750 instance hours a month per workspace | Spins down after 15 min idle; about 1 min to wake; ephemeral disk (no SQLite) |
| **Neon** Postgres | Free plan, no credit card: 100 CU-hours per project, 1 GB storage, scale to zero after 5 min | Cold start on the first query after idle |
| **GHCR** image | Free for public images | Make the package public, or give Render a registry credential |
| **Groq** | Free tier, per-model daily limits | The service's global cost cap keeps you under it |

Steps (once):

1. **Neon:** create a project, copy the connection string (`postgresql://…?sslmode=require`).
2. **GitHub:** push your fork; the `m12 service` workflow pushes `ghcr.io/<you>/mastery-m12-service:<sha>`. In *Packages*, make it public.
3. **Render:** *New → Blueprint* with `apps/m12_service/render.yaml` (set the image URL to your package), or a web service from that image. Set the `sync: false` secrets in the dashboard: `MASTERY_SERVICE_API_KEYS` (generate keys with `python -c "import secrets; print(secrets.token_urlsafe(24))"`), `MASTERY_SERVICE_DATABASE_URL` (Neon), `GROQ_API_KEY`. `MASTERY_SERVICE_PUBLIC=true` refuses to start if you forget the keys.
4. **Deploy hook:** copy it from the service's *Settings* into the GitHub secret `RENDER_DEPLOY_HOOK_URL`; put the public URL in the repository *variable* `M12_PUBLIC_URL`. Every merge to `main` now deploys and waits until `/version` shows the commit.
5. Put the URL in your README, with the demo customer key you are willing to share (its per-user cap limits the damage).

The next cell checks a deployed URL from here. Set `M12_PUBLIC_URL` in `.env` (and, for the 💸 run, `M12_PUBLIC_KEY` with a *customer* key of your deployment). The 💸 run uses your deployment's Groq quota (about 4–6K tokens).

In [ ]:
# 🌐 Check the public deployment: cold start, /version, /ready. (No model call.)
PUBLIC_URL = os.environ.get("M12_PUBLIC_URL", "").strip().rstrip("/")
HAVE_PUBLIC_URL = bool(PUBLIC_URL)
if HAVE_PUBLIC_URL:
    started = time.perf_counter()
    with httpx.Client(base_url=PUBLIC_URL, timeout=120) as pub:
        version = pub.get("/version").json()
        first = time.perf_counter() - started
        ready = pub.get("/ready")
        print(f"first response: {first:.1f} s (≈ 60 s means it was asleep)")
        print("version       :", version)
        print("ready         :", ready.status_code, ready.json().get("problems"))
        print("demo key works:", pub.post("/threads", headers=CUSTOMER).status_code, "(401 expected: public=True)")
else:
    print("⏭️ Skipped: set M12_PUBLIC_URL in .env to check your deployment.")

In [ ]:
# 💸 One LIVE run on the public deployment (uses its Groq quota: about 4–6K tokens).
LIVE_PUBLIC = False          # set True after the 🌐 cell above works
PUBLIC_KEY = os.environ.get("M12_PUBLIC_KEY", "").strip()
if LIVE_PUBLIC and HAVE_PUBLIC_URL and PUBLIC_KEY:
    key = {"Authorization": f"Bearer {PUBLIC_KEY}"}
    with httpx.Client(base_url=PUBLIC_URL, timeout=120) as pub:
        tid = pub.post("/threads", headers=key).json()["thread_id"]
        run = pub.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"}, headers=key).json()
        pub.get(f"/runs/{run['run_id']}/stream", headers=key)
        status = pub.get(f"/runs/{run['run_id']}", headers=key).json()
        print(status["status"], "·", (status["answer"] or "")[:120])
        print({k: status[k] for k in ("ttft_s", "duration_s", "cost_usd")})
else:
    print("⏭️ Skipped: set LIVE_PUBLIC = True and M12_PUBLIC_URL / M12_PUBLIC_KEY in .env.")

**You should now see** (with a deployment) a first response of about a minute if it was asleep, your release and `mode: live`, `ready: 200 []`, and **401** for `demo-customer`. Without the variables both cells print `⏭️ Skipped`.

In [ ]:
# 🧪 Offline: stop this notebook's service.
client.close()
stop_service()
print("stopped:", BASE)

## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Render: "no open port detected" | The image listens on 8000; Render expects `PORT` (default 10000) | Set `PORT=8000` (in `render.yaml`) |
| Render can't pull the image | The GHCR package is private | Make it public, or add a registry credential in Render |
| Deploy hook rejects `imgURL` | Image URL differs from the service's image (only the tag may change) | Use the same `ghcr.io/<you>/mastery-m12-service` path |
| Service refuses to start after deploy | `MASTERY_SERVICE_PUBLIC=true` and demo or short keys | Set real keys in the dashboard (24+ characters) |
| Data gone after a spin-down | SQLite on the instance's ephemeral disk | Neon Postgres in `MASTERY_SERVICE_DATABASE_URL` |
| The UI shows nothing in the browser console's CSP errors | A script or style was added inline | Put it in `app.js` / `style.css` |
| CI is green but the deploy runs old code | `:latest` tag, or the hook without `imgURL` | Deploy the commit's tag; check `/version` |
| `pre-commit` changes every notebook | nbstripout without `--keep-id` | Keep `args: [--keep-id]` |

## ⚠️ Production notes

> ⚠️ **Production note: a free tier gives you a live URL, not production.** Spin-down, one instance, small quotas and a 30-day free Postgres on some providers. For real users: a paid plan with no spin-down, at least two instances, managed Postgres with backups and point-in-time recovery, and budget alerts on every provider.

> ⚠️ **Production note: the kill switch must be faster than a deploy.** `POST /admin/pause` works per process; with several instances, keep the flag in the database or the platform's environment so every instance sees it.

> ⚠️ **Production note: public data only.** The demo uses the synthetic Orbita Cloud customers. Never put real customer data behind a portfolio demo.

## 🏋️ Exercises

1. **(Easy)** Add a "release" badge to the UI that turns orange when `/version` reports `paused: true`.
2. **(Medium)** Use `CanaryRouter` inside the service: build two managers (stable and candidate, e.g. a different `fail_rate` in DEMO) and pick one per user in `start_run`. Tag each run's metadata with which one served it.
3. **(Medium)** Write a rollback criterion for **answer quality**: the share of shadow cases that regress must be 0, and the red-team results must not get worse. Wire it to `shadow_compare`.
4. **(Stretch)** Deploy for real (Part 6), make the `m12 service` and `Eval gate` checks required, open a PR that breaks auth (`use_context=False`), and watch the gate block the merge.

<details>
<summary><b>Show solutions</b></summary>

**1.** In `app.js`, after `fetch("/version")`: `$("version").className = v.paused ? "paused" : "muted"`, and in `style.css` `.paused { color: #b26b00; font-weight: 600; }`. No inline style: the CSP would block it.

**2.** In `create_app`'s lifespan keep `app.state.managers = {"stable": m1, "canary": m2}` and in `start_run` use `runs = app.state.managers[router.route(owner)]`. Threads must stay on one version: route by the thread's owner (sticky) and store the version in the run's trace metadata (`manager.trace_metadata["variant"] = "canary"`).

**3.** `def quality_ok(result): return result["regressions"] == 0 and not result["newly_unblocked"]` — this is `result["ship"]`. Run it in CI next to the eval gate with a sample of last week's inputs (anonymised).

**4.** The PR's `Eval gate` check fails with *blocked by: red team*; with the check required, *Merge* stays disabled until it passes.
</details>

## 🔎 Docs drill (5 min)

On Render's free instances page, after how long without traffic does a free web service spin down, and roughly how long does it take to spin back up?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Render: free instances](https://render.com/docs/free) and find *Spinning down on idle*.
2. **15 minutes** without inbound traffic; spinning back up takes **about one minute**.
3. Compare with the 🌐 cell's "first response" time.

</details>

## ✅ Checkpoint

- [ ] The UI streams an answer, survives a reload mid-answer, and stops a run.
- [ ] A public configuration refuses demo keys; the kill switch and the global cap work.
- [ ] I can read the pipeline: lint → tests → image by commit → smoke → push → deploy → `/version` check.
- [ ] The eval gate exits 1 on the broken refactor; I know how to make it a required check.
- [ ] Shadow comparison found the auth regression; I can explain promote / hold / rollback for a canary.
- [ ] I practised a rollback and know how long it takes.
- [ ] (🌐) My service runs on a public URL, behind real keys, with a cost cap.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Render free instances](https://render.com/docs/free) and [Neon pricing](https://neon.com/pricing) and log in [`docs/research-log.md`](../../docs/research-log.md): the spin-down time, free instance hours, Neon's free compute and storage (2026-10-16: 15 min, 750 hours, 100 CU-hours and 1 GB per project), and the latest major versions of `actions/checkout` and `docker/build-push-action` (2026-10-16: v7 and v7).

## Summary

- The UI is a client of your secured API: streaming with `fetch`, resume with `Last-Event-ID`, a strict CSP.
- A public demo needs real keys, per-user and global cost caps, a kill switch and `/version`.
- CI builds one immutable image per commit, tests it, gates it on evals, and deploys that exact tag.
- Shadow and canary catch regressions before all users see them; rollback criteria are written first.
- A rollback is a redeploy of the previous tag; practise it.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria), then open [`12-project-production-agent-service.ipynb`](project/12-project-production-agent-service.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)